# 03 Channel and timing

This notebook shows where in the call the targets were stated, and for the targets whose figure reached a report only later, how long the call was ahead. It reads `exports/targets_39.csv`, written by `01_call_targets.ipynb`.

In [1]:
# project setup - run first, do not edit except NAME
NAME = "call_only_targets"

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
    subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
                   check=False)
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}")

ok  python 3.13.15  pandas 2.2.3


In [2]:
# folders and helpers shared by the notebooks of this section
import json, re
NB_DIR = ROOT / "notebooks" / "05_call_only_targets"
sys.path.insert(0, str(NB_DIR))
import report_checks as rc

IN = DATA / "call_only_targets"        # delivered result files
REPORTS = DATA / "reports"             # report files, optional (see 00_get_reports)
EXPORTS = NB_DIR / "exports"
EXPORTS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 160, "display.max_rows", 60)

manifest = pd.read_csv(IN / "reports_manifest.csv", keep_default_na=False)
status = rc.report_status(REPORTS, manifest)
print(rc.mode_message(status))

REPORTS PRESENT (29 of 29): every quote is verified against the text of its page.


## Channel

In [3]:
t = pd.read_csv(EXPORTS / "targets_39.csv", keep_default_na=False)
channel = t.groupby(["report_status", "section"]).size().unstack(fill_value=0)
channel

section,prepared,qa
report_status,,
another figure,5,2
no figure,4,0
not mentioned,4,0
same figure,23,0
separate,1,0


In [4]:
only = t[t.report_status.isin(["not mentioned", "no figure"])]
only[["unit", "call", "section", "speaker"]].sort_values("unit")

,unit,call,section,speaker
35,F11,2023-Q4,prepared,Todd Tuckner
36,F12,2023-Q4,prepared,Todd Tuckner
38,F22,2024-Q4,prepared,Todd Tuckner
0,T16,2023-Q3,prepared,Todd Tuckner
1,T25,2023-Q4,prepared,Sergio P. Ermotti
2,T27,2023-Q4,prepared,Sergio P. Ermotti
4,T33,2023-Q4,prepared,Todd Tuckner
5,T68,2024-Q4,prepared,Todd Tuckner


All eight call-only figures were stated in the prepared remarks, not in the Q&A, by the chief financial officer or the chief executive officer. Analysts therefore did not have to ask for them; the figures were volunteered in the call and left out of the written record.

## Lead of the call over the reports

In [5]:
lead = pd.read_csv(IN / "call_first.csv", keep_default_na=False)
lead_view = lead[["unit", "call_quarter", "call_date", "first_report", "report_date", "lead_months", "strength"]]
lead_view.to_csv(EXPORTS / "lead_time.csv", index=False)
lead_view

,unit,call_quarter,call_date,first_report,report_date,lead_months,strength
0,T10,2023-Q2,2023-08-31,2023-Q4,2024-02-06,5.2,confirmed by manual check
1,T17,2023-Q3,2023-11-07,2024-Q2,2024-08-14,9.2,"weak, checked manually"
2,T34,2023-Q4,2024-02-06,2023 annual report,2024-03-28,1.7,"weak, not checked"
3,T36,2023-Q4,2024-02-06,2024-Q1,2024-05-07,3.0,confirmed by manual check
4,T55,2024-Q3,2024-10-30,2024 annual report,2025-03-17,4.5,"weak, not checked"


For five targets the figure appears in a report only after the call. Two cases were confirmed by a manual check: the CET1 guidance of around 14% (5.2 months) and the tax rate of around 40% (3.0 months); the other three are weak, because an equivalent statement exists earlier or the gap is one reporting step.

In [6]:
lead[["unit", "call_quote", "report_quote", "report_location"]]

,unit,call_quote,report_quote,report_location
0,T10,We expect to operate at around 14% CET1 capital ratio over the medium term.,capital ratio of around 14%,"UBS_2023-Q4_report.htm, page 7"
1,T17,"As a reminder, we continue to expect a roughly 5% increase from day-1 effects in 2025 from other final Basel 3 considerations, mainly FRTB.",will lead to an increase of around 5% in UBS Group risk-weighted assets,"UBS_2024-Q2_report.htm, page 8"
2,T34,"Overall, as a result of lower funding needs, diversified and more stable funding sources, tighter issuance spreads relative to 2023 levels, and disciplined ...","Additionally, we expect up to USD 1bn of funding cost","UBS_2023_annual_report.htm, page 46"
3,T36,"The legal entity mergers planned for later this year will resolve a considerable level of this inefficiency, driving down our effective tax rate to around 4...",with our effective tax rate still expected to be around 40% by the end of 2024,"UBS_2024-Q1_report.htm, page 18"
4,T55,By the end of 2026 we aim to have less than 5% remaining.,We aim to achieve a share of below 5% of Group RWA by the end of 2026,"UBS_2024_annual_report.htm, page 52"


In [7]:
# checks against the primary values
assert len(only) == 8 and (only.section == "prepared").all()
assert len(lead) == 5
confirmed = lead[lead.strength.str.startswith("confirmed")].set_index("unit").lead_months.astype(float)
assert confirmed.round(1).to_dict() == {"T10": 5.2, "T36": 3.0}
print("all checks passed")

all checks passed
